# Student Programming Skills Prediction
Multivariate linear regression, from scratch (NumPy) vs scikit-learn.

Run from the project folder (needs `Studentdata.csv` and `student_skills.py`). Works locally and on Colab: upload both files first.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import student_skills as ss

df = ss.load_data("Studentdata.csv")
X, y = ss.build_features(df)
print(X.shape)
y.value_counts().sort_index()

## Split and baseline
The mean-prediction baseline is the bar any model must beat.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=ss.RANDOM_STATE)
ss.baseline_metrics(y_tr, y_te)

## Manual vs scikit-learn OLS

In [ ]:
skl = ss.make_sklearn().fit(X_tr, y_tr)
man = ss.ManualLinearRegression().fit(X_tr, y_tr)
print("coefficients match:", np.allclose(skl.coef_, man.coef_, atol=1e-6))
pd.DataFrame({"sklearn": ss.regression_metrics(y_te, skl.predict(X_te)),
              "manual": ss.regression_metrics(y_te, man.predict(X_te))})

## Ridge (penalty chosen by CV on the training split only)

In [ ]:
grid = [0.1, 1, 3, 10, 30, 100, 300]
scores = {a: ss.cross_validate(lambda: ss.ManualLinearRegression(alpha=a), X_tr, y_tr)["R2"][0] for a in grid}
best = max(scores, key=scores.get)
ridge = ss.ManualLinearRegression(alpha=best).fit(X_tr, y_tr)
print("alpha =", best)
ss.regression_metrics(y_te, ridge.predict(X_te))

## Cross-validated R² (5-fold, all data)

In [ ]:
for name, f in {"OLS": ss.make_sklearn, "Ridge": lambda: ss.ManualLinearRegression(alpha=best)}.items():
    m, s = ss.cross_validate(f, X, y)["R2"]
    print(f"{name}: {m:.3f} +/- {s:.3f}")

## Largest coefficients (ridge)

In [ ]:
c = pd.Series(ridge.coef_, index=X.columns)
c.reindex(c.abs().sort_values(ascending=False).index).head(10)

In [ ]:
rng = np.random.default_rng(0)
plt.scatter(y_te + rng.normal(0, .08, len(y_te)), ridge.predict(X_te), alpha=.7)
plt.plot([0.5, 6.5], [0.5, 6.5], "k--")
plt.xlabel("Actual (jittered)"); plt.ylabel("Predicted"); plt.title("Ridge: actual vs predicted"); plt.show()